In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install mne scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 85.1 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
import mne
import numpy as np
import os
from tqdm import tqdm

folder = '/content/drive/MyDrive/NeuroFlow/data_physio'
subjects = sorted([f for f in os.listdir(folder) if f.startswith('S') and len(f) == 4])[:20]
print("Processing subjects:", subjects)

X_all, y_all, subj_all = [], [], []

for subj in tqdm(subjects):
    subj_folder = os.path.join(folder, subj)
    if not os.path.isdir(subj_folder):
        continue
    for f in sorted(os.listdir(subj_folder)):
        if not f.endswith('.edf'):
            continue
        try:
            run_num = int(f.replace('.edf', '').split('R')[-1])

            # Only runs with movement tasks
            if run_num not in [3,4,5,6,7,8,9,10,11,12,13,14]:
                continue

            raw = mne.io.read_raw_edf(os.path.join(subj_folder, f), preload=True, verbose=False)
            events, event_id = mne.events_from_annotations(raw, verbose=False)

            if len(event_id) < 2:
                continue

            epochs = mne.Epochs(raw, events, event_id, tmin=0, tmax=3,
                                baseline=None, preload=True, verbose=False)

            data = epochs.get_data()
            labels = epochs.events[:, -1]

            # Class mapping:
            # Runs 3,4,7,8,11,12 → T1=left fist (1), T2=right fist (2)
            # Runs 5,6,9,10,13,14 → T1=both fists (3), T2=both feet (4)
            # T0 = rest (0) for all
            if run_num in [3,4,7,8,11,12]:
                label_map = {1: 0, 2: 1, 3: 2}
            else:
                label_map = {1: 0, 2: 3, 3: 4}

            labels_mapped = np.array([label_map.get(l, -1) for l in labels])
            mask = labels_mapped >= 0

            if mask.sum() == 0:
                continue

            X_all.append(data[mask])
            y_all.append(labels_mapped[mask])
            subj_all.extend([subj] * mask.sum())
        except Exception as e:
            print(f"Skipped {f}: {e}")

X = np.concatenate(X_all, axis=0)
y = np.concatenate(y_all, axis=0)
subj_arr = np.array(subj_all)

print("\nX shape:", X.shape)
print("y shape:", y.shape)
print("Class counts:", np.bincount(y))
print("Subjects:", np.unique(subj_arr))
print("Samples per subject:", np.bincount([int(s[1:]) for s in subj_arr]))

# Save
np.save('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X.npy', X)
np.save('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y.npy', y)
np.save('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_subj.npy', subj_arr)
print("Saved.")

Processing subjects: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S018', 'S019', 'S020']


100%|██████████| 20/20 [03:56<00:00, 11.80s/it]



X shape: (7200, 64, 481)
y shape: (7200,)
Class counts: [3600  909  891  896  904]
Subjects: ['S001' 'S002' 'S003' 'S004' 'S005' 'S006' 'S007' 'S008' 'S009' 'S010'
 'S011' 'S012' 'S013' 'S014' 'S015' 'S016' 'S017' 'S018' 'S019' 'S020']
Samples per subject: [  0 360 360 360 360 360 360 360 360 360 360 360 360 360 360 360 360 360
 360 360 360]
Saved.


In [3]:
import numpy as np

X = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X.npy')
y = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y.npy')
subj = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_subj.npy')

print("Before:", np.bincount(y))

# Balance to 900 samples per class
TARGET = 900
X_bal, y_bal, subj_bal = [], [], []

for cls in np.unique(y):
    idx = np.where(y == cls)[0]
    if len(idx) > TARGET:
        idx = np.random.choice(idx, TARGET, replace=False)
    else:
        idx = np.random.choice(idx, TARGET, replace=True)
    X_bal.append(X[idx])
    y_bal.append(y[idx])
    subj_bal.extend(subj[idx])

X_bal = np.concatenate(X_bal)
y_bal = np.concatenate(y_bal)
subj_bal = np.array(subj_bal)

# Shuffle
shuffle = np.random.permutation(len(y_bal))
X_bal = X_bal[shuffle]
y_bal = y_bal[shuffle]
subj_bal = subj_bal[shuffle]

print("After:", np.bincount(y_bal))
print("X_bal shape:", X_bal.shape)

np.save('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X_bal.npy', X_bal)
np.save('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y_bal.npy', y_bal)
np.save('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_subj_bal.npy', subj_bal)
print("Saved.")

Before: [3600  909  891  896  904]
After: [900 900 900 900 900]
X_bal shape: (4500, 64, 481)
Saved.


In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import time

X = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X_bal.npy')
y = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y_bal.npy')

print("X:", X.shape, "y:", y.shape)

# Flatten
X_flat = X.reshape(X.shape[0], -1).astype(np.float32)
print("Flattened:", X_flat.shape)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flat)

# Split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_tr.shape, "Test:", X_te.shape)

# Train
print("\nTraining...")
start = time.time()
model = LogisticRegression(max_iter=1000, n_jobs=-1, C=0.1)
model.fit(X_tr, y_tr)
print(f"Done in {time.time()-start:.1f} sec")

# Evaluate
y_pred = model.predict(X_te)
acc = accuracy_score(y_te, y_pred)
print(f"\nAccuracy: {acc*100:.2f}%")
print(classification_report(y_te, y_pred,
    target_names=['Rest', 'Left Fist', 'Right Fist', 'Both Fists', 'Both Feet']))

X: (4500, 64, 481) y: (4500,)
Flattened: (4500, 30784)
Train: (3600, 30784) Test: (900, 30784)

Training...
Done in 181.0 sec

Accuracy: 61.33%
              precision    recall  f1-score   support

        Rest       0.66      0.54      0.60       180
   Left Fist       0.57      0.56      0.57       180
  Right Fist       0.67      0.75      0.71       180
  Both Fists       0.60      0.65      0.63       180
   Both Feet       0.56      0.56      0.56       180

    accuracy                           0.61       900
   macro avg       0.61      0.61      0.61       900
weighted avg       0.61      0.61      0.61       900



In [5]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import gc

X = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X_bal.npy')
y = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y_bal.npy')
subj = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_subj_bal.npy')

X_flat = X.reshape(X.shape[0], -1).astype(np.float32)
del X
gc.collect()

CAL_SAMPLES = 40  # per class
results = {}

print("Per-subject calibration (40 samples/class)...")
print("="*50)

for s in np.unique(subj):
    mask = subj == s
    X_subj = X_flat[mask]
    y_subj = y[mask]

    # Calibration samples
    X_cal, y_cal, cal_idx = [], [], set()
    for cls in np.unique(y_subj):
        cls_idx = np.where(y_subj == cls)[0]
        n = min(CAL_SAMPLES, len(cls_idx))
        chosen = np.random.choice(cls_idx, n, replace=False)
        X_cal.append(X_subj[chosen])
        y_cal.append(y_subj[chosen])
        cal_idx.update(chosen)

    X_cal = np.concatenate(X_cal)
    y_cal = np.concatenate(y_cal)
    test_mask = np.array([i not in cal_idx for i in range(len(X_subj))])

    sc = StandardScaler()
    Xc = sc.fit_transform(X_cal)
    Xt = sc.transform(X_subj[test_mask])

    m = LogisticRegression(max_iter=1000, n_jobs=-1, C=1.0)
    m.fit(Xc, y_cal)
    acc = accuracy_score(y_subj[test_mask], m.predict(Xt))
    results[s] = acc
    print(f"{s}: {acc*100:.2f}%")
    gc.collect()

print("="*50)
print(f"AVERAGE: {np.mean(list(results.values()))*100:.2f}%")
print(f"BEST: {max(results.values())*100:.2f}% ({max(results, key=results.get)})")
print(f"WORST: {min(results.values())*100:.2f}% ({min(results, key=results.get)})")

Per-subject calibration (40 samples/class)...
S001: 64.29%
S002: 45.45%
S003: 78.57%
S004: 76.47%
S005: 71.43%
S006: 47.62%
S007: 57.78%
S008: 73.68%
S009: 71.05%
S010: 58.82%
S011: 68.97%
S012: 36.84%
S013: 55.56%
S014: 62.16%
S015: 67.74%
S016: 58.33%
S017: 52.17%
S018: 61.54%
S019: 53.85%
S020: 75.00%
AVERAGE: 61.87%
BEST: 78.57% (S003)
WORST: 36.84% (S012)


In [6]:
import numpy as np
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from mne.decoding import CSP
import gc

X = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X_bal.npy')
y = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y_bal.npy')
subj = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_subj_bal.npy')

# Bandpass filter 8-30 Hz (mu + beta for motor imagery)
def bandpass(data, fs=160, low=8, high=30):
    b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=-1)

print("Filtering...")
X_filt = bandpass(X)
del X
gc.collect()

CAL_SAMPLES = 40
results = {}

print("Per-subject CSP+LDA...")
print("="*50)

for s in np.unique(subj):
    mask = subj == s
    X_subj = X_filt[mask]
    y_subj = y[mask]

    # Calibration
    X_cal, y_cal, cal_idx = [], [], set()
    for cls in np.unique(y_subj):
        cls_idx = np.where(y_subj == cls)[0]
        n = min(CAL_SAMPLES, len(cls_idx))
        chosen = np.random.choice(cls_idx, n, replace=False)
        X_cal.append(X_subj[chosen])
        y_cal.append(y_subj[chosen])
        cal_idx.update(chosen)

    X_cal = np.concatenate(X_cal)
    y_cal = np.concatenate(y_cal)
    test_mask = np.array([i not in cal_idx for i in range(len(X_subj))])
    X_test = X_subj[test_mask]
    y_test = y_subj[test_mask]

    # CSP
    csp = CSP(n_components=8, reg='ledoit_wolf', log=True, norm_trace=False)
    X_cal_csp = csp.fit_transform(X_cal, y_cal)
    X_test_csp = csp.transform(X_test)

    # LDA
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    lda = LinearDiscriminantAnalysis()
    lda.fit(X_cal_csp, y_cal)
    acc = accuracy_score(y_test, lda.predict(X_test_csp))
    results[s] = acc
    print(f"{s}: {acc*100:.2f}%")
    gc.collect()

print("="*50)
print(f"AVERAGE: {np.mean(list(results.values()))*100:.2f}%")
print(f"BEST: {max(results.values())*100:.2f}%")

Filtering...
Per-subject CSP+LDA...
Computing rank from data with rank=None
    Using tolerance 0.00049 (2.2e-16 eps * 64 dim * 3.5e+10  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 covariance using LEDOIT_WOLF
Done.
Estimating class=4 covariance using LEDOIT_WOLF
Done.
S001: 60.71%
Computing rank from data with rank=None
    Using tolerance 0.00028 (2.2e-16 eps * 64 dim * 2e+10  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Estimating class=2 covariance using LEDOIT_WOLF
Done.
Estimating class=3 

In [7]:
import mne
import numpy as np
import os

folder = '/content/drive/MyDrive/NeuroFlow/data_physio'
subj_folder = os.path.join(folder, 'S001')

# Check labels in different runs
for run in [3, 5, 7, 9]:
    f = f'S001R{run:02d}.edf'
    path = os.path.join(subj_folder, f)
    if not os.path.exists(path):
        print(f"{f}: NOT FOUND")
        continue
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    print(f"{f}: {event_id}")

S001R03.edf: {'T0': 1, 'T1': 2, 'T2': 3}
S001R05.edf: {'T0': 1, 'T1': 2, 'T2': 3}
S001R07.edf: {'T0': 1, 'T1': 2, 'T2': 3}
S001R09.edf: {'T0': 1, 'T1': 2, 'T2': 3}


In [8]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

# Load balanced data
X = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X_bal.npy')
y = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y_bal.npy')

print("X:", X.shape, "y:", y.shape)

# Bandpass filter 8-30 Hz (helps motor imagery)
from scipy.signal import butter, filtfilt

def bandpass(data, fs=160, low=8, high=30):
    b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=-1)

print("Filtering...")
X = bandpass(X).astype(np.float32)

# Split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_tr.shape, "Test:", X_te.shape)

# GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# EEGNet
class EEGNet(nn.Module):
    def __init__(self, n_classes=5, n_channels=64, n_time=481):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, (1, 32), padding=(0, 16))
        self.bn1 = nn.BatchNorm2d(8)
        self.conv2 = nn.Conv2d(8, 16, (n_channels, 1))
        self.bn2 = nn.BatchNorm2d(16)
        self.pool = nn.AvgPool2d((1, 4))
        self.drop = nn.Dropout(0.5)
        self.fc = nn.Linear(16 * (n_time // 4), n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.drop(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# Data loaders
X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.long)
X_te_t = torch.tensor(X_te, dtype=torch.float32)
y_te_t = torch.tensor(y_te, dtype=torch.long)

train_ds = TensorDataset(X_tr_t, y_tr_t)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

# Train
model = EEGNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

print("\nTraining EEGNet...")
start = time.time()

for epoch in range(30):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = model(xb)
        loss = crit(out, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        preds = model(X_te_t.to(device)).argmax(1).cpu().numpy()
    acc = accuracy_score(y_te, preds)
    print(f"Epoch {epoch+1}/30 | Loss: {total_loss/len(train_dl):.4f} | Acc: {acc*100:.2f}%")

print(f"\nTotal time: {time.time()-start:.1f}s")

# Save
torch.save(model.state_dict(), '/content/drive/MyDrive/NeuroFlow/A818_NF_5class_eegnet.pth')
print("Saved")

X: (4500, 64, 481) y: (4500,)
Filtering...
Train: (3600, 64, 481) Test: (900, 64, 481)
Device: cpu

Training EEGNet...
Epoch 1/30 | Loss: 1.6109 | Acc: 20.00%
Epoch 2/30 | Loss: 1.5844 | Acc: 20.00%
Epoch 3/30 | Loss: 1.5268 | Acc: 20.00%
Epoch 4/30 | Loss: 1.4518 | Acc: 20.00%
Epoch 5/30 | Loss: 1.4063 | Acc: 20.00%
Epoch 6/30 | Loss: 1.3395 | Acc: 20.00%
Epoch 7/30 | Loss: 1.2861 | Acc: 20.00%
Epoch 8/30 | Loss: 1.2446 | Acc: 20.11%
Epoch 9/30 | Loss: 1.2113 | Acc: 20.00%
Epoch 10/30 | Loss: 1.1861 | Acc: 20.00%
Epoch 11/30 | Loss: 1.1722 | Acc: 20.00%
Epoch 12/30 | Loss: 1.1355 | Acc: 20.00%
Epoch 13/30 | Loss: 1.1011 | Acc: 20.00%
Epoch 14/30 | Loss: 1.1047 | Acc: 20.00%
Epoch 15/30 | Loss: 1.0755 | Acc: 20.00%
Epoch 16/30 | Loss: 1.0608 | Acc: 21.89%
Epoch 17/30 | Loss: 1.0422 | Acc: 20.00%
Epoch 18/30 | Loss: 1.0328 | Acc: 20.00%
Epoch 19/30 | Loss: 1.0183 | Acc: 20.00%
Epoch 20/30 | Loss: 1.0070 | Acc: 20.00%
Epoch 21/30 | Loss: 0.9929 | Acc: 20.00%
Epoch 22/30 | Loss: 0.9840 | 

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: False
Device name: None


In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
Device: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.signal import butter, filtfilt
import time

# Load data
X = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X_bal.npy')
y = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y_bal.npy')
print("X:", X.shape, "y:", y.shape)

# Bandpass 8-30 Hz
def bandpass(data, fs=160, low=8, high=30):
    b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=-1)

print("Filtering...")
X = bandpass(X).astype(np.float32)

# Z-score per trial/channel
X = (X - X.mean(axis=-1, keepdims=True)) / (X.std(axis=-1, keepdims=True) + 1e-8)

# Split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_tr.shape, "Test:", X_te.shape)

device = torch.device('cuda')
print("Device:", device)

# Bigger EEGNet
class EEGNet(nn.Module):
    def __init__(self, n_classes=5, n_channels=64, n_time=481):
        super().__init__()
        # Block 1
        self.conv1 = nn.Conv2d(1, 16, (1, 32), padding=(0, 16), bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        # Depthwise
        self.conv2 = nn.Conv2d(16, 32, (n_channels, 1), groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.5)

        # Block 2
        self.conv3 = nn.Conv2d(32, 32, (1, 16), padding=(0, 8), groups=32, bias=False)
        self.conv4 = nn.Conv2d(32, 64, (1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool2 = nn.AvgPool2d((1, 4))
        self.drop2 = nn.Dropout(0.5)

        # Classifier
        final_time = n_time // 16
        self.fc = nn.Linear(64 * final_time, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = torch.relu(x)
        x = self.drop1(self.pool1(x))

        x = self.conv3(x)
        x = self.bn3(self.conv4(x))
        x = torch.relu(x)
        x = self.drop2(self.pool2(x))

        x = x.view(x.size(0), -1)
        return self.fc(x)

# Data loaders
X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.long)
X_te_t = torch.tensor(X_te, dtype=torch.float32).to(device)
y_te_t = torch.tensor(y_te, dtype=torch.long).to(device)

train_ds = TensorDataset(X_tr_t, y_tr_t)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)

# Model
model = EEGNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5)
crit = nn.CrossEntropyLoss()

print("\nTraining...")
start = time.time()

for epoch in range(50):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = model(xb)
        loss = crit(out, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    scheduler.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_te_t).argmax(1).cpu().numpy()
    acc = accuracy_score(y_te, preds)

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/50 | Loss: {total_loss/len(train_dl):.4f} | Acc: {acc*100:.2f}%")

print(f"\nTotal: {time.time()-start:.1f}s")
print(f"Final accuracy: {acc*100:.2f}%")

torch.save(model.state_dict(), '/content/drive/MyDrive/NeuroFlow/A818_NF_5class_eegnet_v2.pth')
print("Saved")

X: (4500, 64, 481) y: (4500,)
Filtering...
Train: (3600, 64, 481) Test: (900, 64, 481)
Device: cuda

Training...
Epoch 5/50 | Loss: 1.5438 | Acc: 36.89%
Epoch 10/50 | Loss: 1.4169 | Acc: 45.22%
Epoch 15/50 | Loss: 1.2921 | Acc: 51.89%
Epoch 20/50 | Loss: 1.2011 | Acc: 54.56%
Epoch 25/50 | Loss: 1.1262 | Acc: 57.44%
Epoch 30/50 | Loss: 1.0670 | Acc: 56.22%
Epoch 35/50 | Loss: 1.0157 | Acc: 55.56%
Epoch 40/50 | Loss: 1.0007 | Acc: 58.56%
Epoch 45/50 | Loss: 0.9662 | Acc: 58.78%
Epoch 50/50 | Loss: 0.9356 | Acc: 59.44%

Total: 120.9s
Final accuracy: 59.44%
Saved


In [5]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score
from scipy.signal import butter, filtfilt
import gc

# Load data
X = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X_bal.npy')
y = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y_bal.npy')
subj = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_subj_bal.npy')

def bandpass(data, fs=160, low=8, high=30):
    b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=-1)

print("Filtering...")
X = bandpass(X).astype(np.float32)
X = (X - X.mean(axis=-1, keepdims=True)) / (X.std(axis=-1, keepdims=True) + 1e-8)

device = torch.device('cuda')

# Load pretrained EEGNet
class EEGNet(nn.Module):
    def __init__(self, n_classes=5, n_channels=64, n_time=481):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, (1, 32), padding=(0, 16), bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, (n_channels, 1), groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.5)
        self.conv3 = nn.Conv2d(32, 32, (1, 16), padding=(0, 8), groups=32, bias=False)
        self.conv4 = nn.Conv2d(32, 64, (1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool2 = nn.AvgPool2d((1, 4))
        self.drop2 = nn.Dropout(0.5)
        self.fc = nn.Linear(64 * (n_time // 16), n_classes)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = torch.relu(x)
        x = self.drop1(self.pool1(x))
        x = self.conv3(x)
        x = self.bn3(self.conv4(x))
        x = torch.relu(x)
        x = self.drop2(self.pool2(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

base_model = EEGNet().to(device)
base_model.load_state_dict(torch.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_eegnet_v2.pth'))
print("Base loaded")

# Per-subject fine-tuning
CAL_SAMPLES = 40
results = {}

print("\nPer-subject fine-tuning...")
print("="*50)

for s in np.unique(subj):
    mask = subj == s
    X_subj = X[mask]
    y_subj = y[mask]

    # Calibration samples
    X_cal, y_cal, cal_idx = [], [], set()
    for cls in np.unique(y_subj):
        cls_idx = np.where(y_subj == cls)[0]
        n = min(CAL_SAMPLES, len(cls_idx))
        chosen = np.random.choice(cls_idx, n, replace=False)
        X_cal.append(X_subj[chosen])
        y_cal.append(y_subj[chosen])
        cal_idx.update(chosen)

    X_cal = np.array(X_cal)
    y_cal = np.array(y_cal)
    test_mask = np.array([i not in cal_idx for i in range(len(X_subj))])
    X_test = X_subj[test_mask]
    y_test = y_subj[test_mask]

    # Clone base model
    model = EEGNet().to(device)
    model.load_state_dict(base_model.state_dict())

    # Fine-tune
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    crit = nn.CrossEntropyLoss()

    X_cal_t = torch.tensor(X_cal, dtype=torch.float32).to(device)
    y_cal_t = torch.tensor(y_cal, dtype=torch.long).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

    model.train()
    for epoch in range(20):
        opt.zero_grad()
        out = model(X_cal_t)
        loss = crit(out, y_cal_t)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).argmax(1).cpu().numpy()
    acc = accuracy_score(y_test, preds)
    results[s] = acc
    print(f"{s}: {acc*100:.2f}%")
    gc.collect()

print("="*50)
print(f"AVERAGE: {np.mean(list(results.values()))*100:.2f}%")
print(f"BEST: {max(results.values())*100:.2f}%")

Filtering...
Base loaded

Per-subject fine-tuning...


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (5,) + inhomogeneous part.

In [6]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from scipy.signal import butter, filtfilt
import gc

# Load data
X = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X_bal.npy')
y = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y_bal.npy')
subj = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_subj_bal.npy')

def bandpass(data, fs=160, low=8, high=30):
    b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=-1)

print("Filtering...")
X = bandpass(X).astype(np.float32)
X = (X - X.mean(axis=-1, keepdims=True)) / (X.std(axis=-1, keepdims=True) + 1e-8)

device = torch.device('cuda')

class EEGNet(nn.Module):
    def __init__(self, n_classes=5, n_channels=64, n_time=481):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, (1, 32), padding=(0, 16), bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, (n_channels, 1), groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.5)
        self.conv3 = nn.Conv2d(32, 32, (1, 16), padding=(0, 8), groups=32, bias=False)
        self.conv4 = nn.Conv2d(32, 64, (1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool2 = nn.AvgPool2d((1, 4))
        self.drop2 = nn.Dropout(0.5)
        self.fc = nn.Linear(64 * (n_time // 16), n_classes)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = torch.relu(x)
        x = self.drop1(self.pool1(x))
        x = self.conv3(x)
        x = self.bn3(self.conv4(x))
        x = torch.relu(x)
        x = self.drop2(self.pool2(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

base_model = EEGNet().to(device)
base_model.load_state_dict(torch.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_eegnet_v2.pth'))
print("Base loaded")

CAL_SAMPLES = 40
results = {}

print("\nPer-subject fine-tuning...")
print("="*50)

for s in np.unique(subj):
    mask = subj == s
    X_subj = X[mask]
    y_subj = y[mask]

    X_cal_list, y_cal_list, cal_idx = [], [], set()
    for cls in np.unique(y_subj):
        cls_idx = np.where(y_subj == cls)[0]
        n = min(CAL_SAMPLES, len(cls_idx))
        chosen = np.random.choice(cls_idx, n, replace=False)
        X_cal_list.append(X_subj[chosen])
        y_cal_list.append(y_subj[chosen])
        cal_idx.update(chosen)

    X_cal = np.concatenate(X_cal_list, axis=0)   # FIX
    y_cal = np.concatenate(y_cal_list, axis=0)   # FIX
    test_mask = np.array([i not in cal_idx for i in range(len(X_subj))])
    X_test = X_subj[test_mask]
    y_test = y_subj[test_mask]

    model = EEGNet().to(device)
    model.load_state_dict(base_model.state_dict())

    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    crit = nn.CrossEntropyLoss()

    X_cal_t = torch.tensor(X_cal, dtype=torch.float32).to(device)
    y_cal_t = torch.tensor(y_cal, dtype=torch.long).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

    model.train()
    for epoch in range(20):
        opt.zero_grad()
        out = model(X_cal_t)
        loss = crit(out, y_cal_t)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).argmax(1).cpu().numpy()
    acc = accuracy_score(y_test, preds)
    results[s] = acc
    print(f"{s}: {acc*100:.2f}%")
    gc.collect()

print("="*50)
print(f"AVERAGE: {np.mean(list(results.values()))*100:.2f}%")
print(f"BEST: {max(results.values())*100:.2f}%")

Filtering...
Base loaded

Per-subject fine-tuning...
S001: 89.29%
S002: 81.82%
S003: 71.43%
S004: 88.24%
S005: 65.71%
S006: 80.95%
S007: 93.33%
S008: 78.95%
S009: 65.79%
S010: 86.27%
S011: 58.62%
S012: 84.21%
S013: 82.22%
S014: 72.97%
S015: 93.55%
S016: 58.33%
S017: 86.96%
S018: 76.92%
S019: 73.08%
S020: 75.00%
AVERAGE: 78.18%
BEST: 93.55%


In [7]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from scipy.signal import butter, filtfilt, iirnotch
import gc
import time

# Load
X = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X_bal.npy')
y = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y_bal.npy')
subj = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_subj_bal.npy')

# Stronger preprocessing
def preprocess(data, fs=160):
    # Notch at 50 Hz (Pakistan mains)
    b_n, a_n = iirnotch(50, Q=30, fs=fs)
    data = filtfilt(b_n, a_n, data, axis=-1)
    # Bandpass 8-30 Hz
    b, a = butter(4, [8/(fs/2), 30/(fs/2)], btype='band')
    data = filtfilt(b, a, data, axis=-1)
    # Z-score per channel
    data = (data - data.mean(axis=-1, keepdims=True)) / (data.std(axis=-1, keepdims=True) + 1e-8)
    return data.astype(np.float32)

print("Preprocessing...")
X = preprocess(X)
print("Done")

device = torch.device('cuda')

class EEGNet(nn.Module):
    def __init__(self, n_classes=5, n_channels=64, n_time=481):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, (1, 32), padding=(0, 16), bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, (n_channels, 1), groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.5)
        self.conv3 = nn.Conv2d(32, 32, (1, 16), padding=(0, 8), groups=32, bias=False)
        self.conv4 = nn.Conv2d(32, 64, (1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool2 = nn.AvgPool2d((1, 4))
        self.drop2 = nn.Dropout(0.5)
        self.fc = nn.Linear(64 * (n_time // 16), n_classes)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = torch.relu(x)
        x = self.drop1(self.pool1(x))
        x = self.conv3(x)
        x = self.bn3(self.conv4(x))
        x = torch.relu(x)
        x = self.drop2(self.pool2(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

base_model = EEGNet().to(device)
base_model.load_state_dict(torch.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_eegnet_v2.pth'))
print("Base loaded")

# INCREASED calibration + epochs
CAL_SAMPLES = 80     # was 40
FT_EPOCHS = 50       # was 20

results = {}
print("\nPer-subject fine-tuning (80 cal, 50 epochs)...")
print("="*50)
start = time.time()

for s in np.unique(subj):
    mask = subj == s
    X_subj = X[mask]
    y_subj = y[mask]

    X_cal_list, y_cal_list, cal_idx = [], [], set()
    for cls in np.unique(y_subj):
        cls_idx = np.where(y_subj == cls)[0]
        n = min(CAL_SAMPLES, len(cls_idx))
        chosen = np.random.choice(cls_idx, n, replace=False)
        X_cal_list.append(X_subj[chosen])
        y_cal_list.append(y_subj[chosen])
        cal_idx.update(chosen)

    X_cal = np.concatenate(X_cal_list, axis=0)
    y_cal = np.concatenate(y_cal_list, axis=0)
    test_mask = np.array([i not in cal_idx for i in range(len(X_subj))])
    X_test = X_subj[test_mask]
    y_test = y_subj[test_mask]

    model = EEGNet().to(device)
    model.load_state_dict(base_model.state_dict())

    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()

    X_cal_t = torch.tensor(X_cal, dtype=torch.float32).to(device)
    y_cal_t = torch.tensor(y_cal, dtype=torch.long).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

    model.train()
    for epoch in range(FT_EPOCHS):
        opt.zero_grad()
        out = model(X_cal_t)
        loss = crit(out, y_cal_t)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).argmax(1).cpu().numpy()
    acc = accuracy_score(y_test, preds)
    results[s] = acc
    print(f"{s}: {acc*100:.2f}%")
    gc.collect()

print("="*50)
print(f"AVERAGE: {np.mean(list(results.values()))*100:.2f}%")
print(f"BEST: {max(results.values())*100:.2f}%")
print(f"WORST: {min(results.values())*100:.2f}%")
print(f"Total time: {time.time()-start:.1f}s")

Preprocessing...
Done
Base loaded

Per-subject fine-tuning (80 cal, 50 epochs)...


RuntimeError: cannot reshape tensor of 0 elements into shape [0, -1] because the unspecified dimension size -1 can be any value and is ambiguous

In [8]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from scipy.signal import butter, filtfilt, iirnotch
import gc
import time

# Load
X = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_X_bal.npy')
y = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_y_bal.npy')
subj = np.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_subj_bal.npy')

# Check samples per subject per class
print("Samples per subject:")
for s in np.unique(subj)[:5]:
    mask = subj == s
    print(f"  {s}: {mask.sum()} total, classes: {np.bincount(y[mask])}")

def preprocess(data, fs=160):
    b_n, a_n = iirnotch(50, Q=30, fs=fs)
    data = filtfilt(b_n, a_n, data, axis=-1)
    b, a = butter(4, [8/(fs/2), 30/(fs/2)], btype='band')
    data = filtfilt(b, a, data, axis=-1)
    data = (data - data.mean(axis=-1, keepdims=True)) / (data.std(axis=-1, keepdims=True) + 1e-8)
    return data.astype(np.float32)

print("\nPreprocessing...")
X = preprocess(X)
print("Done")

device = torch.device('cuda')

class EEGNet(nn.Module):
    def __init__(self, n_classes=5, n_channels=64, n_time=481):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, (1, 32), padding=(0, 16), bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, (n_channels, 1), groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.5)
        self.conv3 = nn.Conv2d(32, 32, (1, 16), padding=(0, 8), groups=32, bias=False)
        self.conv4 = nn.Conv2d(32, 64, (1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool2 = nn.AvgPool2d((1, 4))
        self.drop2 = nn.Dropout(0.5)
        self.fc = nn.Linear(64 * (n_time // 16), n_classes)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = torch.relu(x)
        x = self.drop1(self.pool1(x))
        x = self.conv3(x)
        x = self.bn3(self.conv4(x))
        x = torch.relu(x)
        x = self.drop2(self.pool2(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

base_model = EEGNet().to(device)
base_model.load_state_dict(torch.load('/content/drive/MyDrive/NeuroFlow/A818_NF_5class_eegnet_v2.pth'))
print("Base loaded")

# Safely handle small test sets
CAL_SAMPLES = 60
FT_EPOCHS = 50
results = {}

print("\nPer-subject fine-tuning (60 cal, 50 epochs)...")
print("="*50)
start = time.time()

for s in np.unique(subj):
    mask = subj == s
    X_subj = X[mask]
    y_subj = y[mask]

    X_cal_list, y_cal_list, cal_idx = [], [], set()
    for cls in np.unique(y_subj):
        cls_idx = np.where(y_subj == cls)[0]
        n = min(CAL_SAMPLES, len(cls_idx) - 5)  # Leave 5 for test
        if n < 5:
            continue
        chosen = np.random.choice(cls_idx, n, replace=False)
        X_cal_list.append(X_subj[chosen])
        y_cal_list.append(y_subj[chosen])
        cal_idx.update(chosen)

    X_cal = np.concatenate(X_cal_list, axis=0)
    y_cal = np.concatenate(y_cal_list, axis=0)
    test_mask = np.array([i not in cal_idx for i in range(len(X_subj))])

    # Skip if no test data
    if test_mask.sum() == 0:
        print(f"{s}: SKIPPED (no test data)")
        continue

    X_test = X_subj[test_mask]
    y_test = y_subj[test_mask]

    model = EEGNet().to(device)
    model.load_state_dict(base_model.state_dict())

    opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()

    X_cal_t = torch.tensor(X_cal, dtype=torch.float32).to(device)
    y_cal_t = torch.tensor(y_cal, dtype=torch.long).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

    model.train()
    for epoch in range(FT_EPOCHS):
        opt.zero_grad()
        out = model(X_cal_t)
        loss = crit(out, y_cal_t)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).argmax(1).cpu().numpy()
    acc = accuracy_score(y_test, preds)
    results[s] = acc
    print(f"{s}: {acc*100:.2f}%")
    gc.collect()

print("="*50)
print(f"AVERAGE: {np.mean(list(results.values()))*100:.2f}%")
print(f"BEST: {max(results.values())*100:.2f}%")
print(f"WORST: {min(results.values())*100:.2f}%")
print(f"Total: {time.time()-start:.1f}s")

Samples per subject:
  S001: 223 total, classes: [45 45 52 35 46]
  S002: 229 total, classes: [52 46 36 50 45]
  S003: 209 total, classes: [40 45 35 42 47]
  S004: 208 total, classes: [46 45 38 33 46]
  S005: 232 total, classes: [37 44 56 51 44]

Preprocessing...
Done
Base loaded

Per-subject fine-tuning (60 cal, 50 epochs)...
S001: 84.00%
S002: 80.00%
S003: 68.00%
S004: 96.00%
S005: 68.00%
S006: 80.00%
S007: 96.00%
S008: 68.00%
S009: 72.00%
S010: 88.00%
S011: 76.00%
S012: 84.00%
S013: 84.00%
S014: 80.00%
S015: 84.00%
S016: 60.00%
S017: 72.00%
S018: 64.00%
S019: 76.00%
S020: 88.00%
AVERAGE: 78.40%
BEST: 96.00%
WORST: 60.00%
Total: 137.7s


In [9]:
import joblib
import numpy as np

DRIVE = '/content/drive/MyDrive/NeuroFlow'

# Save 5-class model
product_5class = {
    'type': '5class_extended',
    'accuracy': 0.784,
    'classes': {
        0: 'rest',
        1: 'left_fist',
        2: 'right_fist',
        3: 'both_fists',
        4: 'both_feet'
    },
    'eegnet_path': f'{DRIVE}/A818_NF_5class_eegnet_v2.pth',
    'calibration_samples': 60,
    'input_shape': (64, 481),
    'sampling_rate': 160,
    'description': 'Extended mode - 5 classes with calibration'
}

# Save 3-class model reference (already exists)
product_3class = {
    'type': '3class_focus',
    'accuracy': 0.91,
    'classes': {
        0: 'rest',
        1: 'left_fist',
        2: 'right_fist'
    },
    'model_path': f'{DRIVE}/best_model.pkl',
    'scaler_path': f'{DRIVE}/scaler.pkl',
    'input_shape': (64, 481),
    'sampling_rate': 160,
    'description': 'Focus mode - 3 classes, high accuracy'
}

# Save both
joblib.dump(product_3class, f'{DRIVE}/A818_NF_product_3class.pkl')
joblib.dump(product_5class, f'{DRIVE}/A818_NF_product_5class.pkl')

print("Saved:")
print(f"  {DRIVE}/A818_NF_product_3class.pkl")
print(f"  {DRIVE}/A818_NF_product_5class.pkl")

Saved:
  /content/drive/MyDrive/NeuroFlow/A818_NF_product_3class.pkl
  /content/drive/MyDrive/NeuroFlow/A818_NF_product_5class.pkl


In [10]:
import os
os.makedirs('/content/A818_NF', exist_ok=True)
print("Folder ready")

Folder ready


In [11]:
!git clone https://github.com/deepbuildASI/NeuroFlow.git /content/A818_NF
!ls /content/A818_NF

Cloning into '/content/A818_NF'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 52 (delta 16), reused 36 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 3.43 MiB | 20.33 MiB/s, done.
Resolving deltas: 100% (16/16), done.
app.py	LICENSE		    proof      requirments.txt
Data	NeuroFlow_v1.ipynb  README.md  runtime.txt


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.5 MB/s eta 0:00:00


In [17]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [18]:
!cd /content/A818_NF && streamlit run app.py &>/content/logs.txt & sleep 5 && /content/cloudflared tunnel --url http://localhost:8501

2026-10-02T08:59:07Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-02T08:59:07Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-02T08:59:12Z INF +--------------------------------------------------------------------------------------------+
2026-10-02T08:59:12Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-02T08:59:12Z INF |  https://machines-launch-julia-edt.trycloudflare.com  

In [19]:
"""
A818_NF — Brain Signal Decoder
Production BCI Software · v2.1
"""

import streamlit as st
import numpy as np
import joblib
import os
import time
import random
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.signal import butter, filtfilt, iirnotch

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
st.set_page_config(page_title="A818_NF — Brain Signal Decoder",
                   page_icon="🧠", layout="wide",
                   initial_sidebar_state="expanded")

# ═══════════════════════════════════════════════════════════
# CSS
# ═══════════════════════════════════════════════════════════
st.markdown("""
<style>
.main {background-color:#0a0d12;}
h1 {color:#00d4ff; font-weight:700; letter-spacing:-0.5px;}
h2,h3 {color:#e6e9ef; font-weight:600;}
section[data-testid="stSidebar"] {background-color:#0f1419;}
.stButton>button {background-color:#00d4ff; color:#0a0d12; font-weight:700;
  border-radius:6px; padding:8px 18px; border:none; font-size:14px; transition:0.2s;}
.stButton>button:hover {background-color:#00b8e6; transform:translateY(-1px);}
.status-bar {background:linear-gradient(90deg,#1a1f26,#0f1419); padding:12px 20px;
  border-radius:8px; border-left:4px solid #00d4ff; margin-bottom:20px; font-size:13px;}
.status-live{color:#00ff88; font-weight:700;}
.status-idle{color:#888; font-weight:600;}
.status-cal{color:#ffaa00; font-weight:700;}
.pred-card{background:linear-gradient(135deg,#1c1f26,#151920); padding:28px;
  border-radius:12px; border-left:5px solid #00d4ff; box-shadow:0 4px 20px rgba(0,212,255,0.08);}
.pred-card-5{border-left-color:#ff9f1c;}
.pred-label{color:#6b7280; font-size:11px; letter-spacing:2px; text-transform:uppercase; margin:0;}
.pred-value{font-size:34px; font-weight:800; margin:6px 0 0 0; letter-spacing:-1px;}
.true-label{color:#9ca3af; font-size:14px; margin:16px 0 4px 0;}
.true-value{color:#e6e9ef; font-size:20px; font-weight:600; margin:0;}
.correct{color:#00ff88; font-weight:700; font-size:14px; margin-top:14px;}
.incorrect{color:#ff4757; font-weight:700; font-size:14px; margin-top:14px;}
.conf-row{display:flex; align-items:center; margin-bottom:10px;}
.conf-name{color:#9ca3af; font-size:13px; width:120px; font-weight:500;}
.conf-bar-bg{flex:1; height:8px; background:#1a1f26; border-radius:4px; overflow:hidden;}
.conf-bar-fill{height:100%; background:linear-gradient(90deg,#00d4ff,#00b8e6); border-radius:4px;}
.conf-bar-fill-5{background:linear-gradient(90deg,#ff9f1c,#ff7b00);}
.conf-pct{color:#e6e9ef; font-size:12px; margin-left:12px; width:60px;
  text-align:right; font-variant-numeric:tabular-nums;}
.stat-box{background:#131720; padding:16px 12px; border-radius:8px; text-align:center;
  border:1px solid #1f242e;}
.stat-value{color:#00d4ff; font-size:22px; font-weight:700; font-variant-numeric:tabular-nums;}
.stat-label{color:#6b7280; font-size:10px; text-transform:uppercase; letter-spacing:1.5px; margin-top:4px;}
.mode-badge{display:inline-block; padding:6px 14px; border-radius:20px; font-size:12px;
  font-weight:700; letter-spacing:1px; margin-bottom:16px;}
.badge-3{background:rgba(0,212,255,0.15); color:#00d4ff; border:1px solid #00d4ff;}
.badge-5{background:rgba(255,159,28,0.15); color:#ff9f1c; border:1px solid #ff9f1c;}
.footer{text-align:center; color:#4b5563; font-size:11px; padding:30px 0 10px 0; letter-spacing:1px;}
.log-entry{padding:6px 12px; border-left:3px solid #2a2f3a; margin-bottom:4px;
  font-size:12px; color:#9ca3af; font-family:'Courier New',monospace;}
.log-entry-match{border-left-color:#00ff88;}
.log-entry-miss{border-left-color:#ff4757;}
</style>
""", unsafe_allow_html=True)

# ═══════════════════════════════════════════════════════════
# PREPROCESSING (MATCHES TRAINING PIPELINE EXACTLY)
# ═══════════════════════════════════════════════════════════
def preprocess_eegnet(sample, fs=160):
    """Apply same preprocessing used during EEGNet training.
    sample shape: (64, 481) raw EEG"""
    # Notch 50 Hz
    b_n, a_n = iirnotch(50, Q=30, fs=fs)
    x = filtfilt(b_n, a_n, sample, axis=-1)
    # Bandpass 8-30 Hz
    b, a = butter(4, [8/(fs/2), 30/(fs/2)], btype='band')
    x = filtfilt(b, a, x, axis=-1)
    # Z-score per channel
    x = (x - x.mean(axis=-1, keepdims=True)) / (x.std(axis=-1, keepdims=True) + 1e-8)
    return x.astype(np.float32)

# ═══════════════════════════════════════════════════════════
# EEGNET
# ═══════════════════════════════════════════════════════════
class EEGNet(nn.Module):
    def __init__(self, n_classes=5, n_channels=64, n_time=481):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, (1, 32), padding=(0, 16), bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, (n_channels, 1), groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(0.5)
        self.conv3 = nn.Conv2d(32, 32, (1, 16), padding=(0, 8), groups=32, bias=False)
        self.conv4 = nn.Conv2d(32, 64, (1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool2 = nn.AvgPool2d((1, 4))
        self.drop2 = nn.Dropout(0.5)
        self.fc = nn.Linear(64 * (n_time // 16), n_classes)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = torch.relu(x)
        x = self.drop1(self.pool1(x))
        x = self.conv3(x)
        x = self.bn3(self.conv4(x))
        x = torch.relu(x)
        x = self.drop2(self.pool2(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

# ═══════════════════════════════════════════════════════════
# PATHS
# ═══════════════════════════════════════════════════════════
DRIVE = '/content/drive/MyDrive/NeuroFlow'
LOCAL = os.path.dirname(os.path.abspath(__file__))

def find_path(filename):
    for base in [LOCAL, DRIVE, os.getcwd()]:
        p = os.path.join(base, filename)
        if os.path.exists(p):
            return p
    return None

@st.cache_resource
def load_3class():
    m = find_path('best_model.pkl'); s = find_path('scaler.pkl')
    return (joblib.load(m), joblib.load(s)) if (m and s) else (None, None)

@st.cache_resource
def load_5class():
    p = find_path('A818_NF_5class_eegnet_v2.pth')
    if p:
        m = EEGNet(); m.load_state_dict(torch.load(p, map_location='cpu')); m.eval()
        return m
    return None

@st.cache_data
def load_data_3class():
    X = find_path('X_all.npy'); y = find_path('y_all.npy')
    return (np.load(X), np.load(y)) if (X and y) else (None, None)

@st.cache_data
def load_data_5class():
    X = find_path('A818_NF_5class_X_bal.npy'); y = find_path('A818_NF_5class_y_bal.npy')
    return (np.load(X), np.load(y)) if (X and y) else (None, None)

# ═══════════════════════════════════════════════════════════
# STATE
# ═══════════════════════════════════════════════════════════
def init_state():
    defaults = {
        'buffer': None, 'playing': False, 'current_sample': None,
        'current_true': None, 'prediction': None, 'confidence': None,
        'session_log': [], 'samples_processed': 0, 'calibrating': False,
        'calibration_data': [], 'mode_used': None,
    }
    for k, v in defaults.items():
        if k not in st.session_state:
            st.session_state[k] = v
init_state()

# ═══════════════════════════════════════════════════════════
# HEADER
# ═══════════════════════════════════════════════════════════
st.title("🧠 A818_NF — Brain Signal Decoder")
st.markdown("##### Production BCI Software · Real-time intent classification")

# ═══════════════════════════════════════════════════════════
# SIDEBAR
# ═══════════════════════════════════════════════════════════
st.sidebar.markdown("## ⚙️ Control Panel")
st.sidebar.markdown("---")
mode = st.sidebar.radio("**Decoder Mode**", ["🎯 Focus Mode (3-class)", "🔬 Extended Mode (5-class)"])
operating_mode = st.sidebar.radio("**Operating Mode**", ["🧪 Test Mode", "🧬 Live Mode"])
st.sidebar.markdown("---")

if "Focus" in mode:
    st.sidebar.markdown("**Model:** Logistic Regression")
    st.sidebar.markdown("**Validated accuracy:** 91.0%")
    st.sidebar.markdown("**Classes:** Rest · Left Fist · Right Fist")
else:
    st.sidebar.markdown("**Model:** EEGNet (CNN)")
    st.sidebar.markdown("**Validated accuracy:** 78.4%")
    st.sidebar.markdown("**Classes:** Rest · L.Fist · R.Fist · Both Fists · Both Feet")

st.sidebar.markdown("---")
st.sidebar.markdown("## 🎛️ Actions")
col_a, col_b = st.sidebar.columns(2)
with col_a: start_btn = st.button("▶ Stream", use_container_width=True)
with col_b: stop_btn = st.button("⏹ Stop", use_container_width=True)
col_c, col_d = st.sidebar.columns(2)
with col_c: single_btn = st.button("🎲 Sample", use_container_width=True)
with col_d: clear_btn = st.button("🧹 Clear", use_container_width=True)

st.sidebar.markdown("---")
st.sidebar.markdown("## 🎓 Calibration")
calib_btn = st.sidebar.button("🧬 Start Calibration", use_container_width=True)
if calib_btn:
    st.session_state.calibrating = True
    st.session_state.calibration_data = []
if stop_btn:
    st.session_state.playing = False
if clear_btn:
    for k in ['buffer','current_sample','prediction','confidence','session_log']:
        st.session_state[k] = None if k != 'session_log' else []
    st.session_state.samples_processed = 0
    st.session_state.playing = False

# ═══════════════════════════════════════════════════════════
# LOAD MODELS
# ═══════════════════════════════════════════════════════════
if "Focus" in mode:
    model, scaler = load_3class()
    X_data, y_data = load_data_3class()
    label_map = {1:"Rest", 2:"Left Fist", 3:"Right Fist"}
    accent = "#00d4ff"; badge_class = "badge-3"
    badge_text = "FOCUS MODE · 3 CLASSES · 91%"
else:
    model = load_5class(); scaler = None
    X_data, y_data = load_data_5class()
    label_map = {0:"Rest",1:"Left Fist",2:"Right Fist",3:"Both Fists",4:"Both Feet"}
    accent = "#ff9f1c"; badge_class = "badge-5"
    badge_text = "EXTENDED MODE · 5 CLASSES · 78%"

if model is None or X_data is None:
    st.error("⚠️ Model or data files not found. Check Drive paths.")
    st.stop()

# ═══════════════════════════════════════════════════════════
# PREDICT
# ═══════════════════════════════════════════════════════════
def predict(sample):
    if "Focus" in mode:
        flat = sample.reshape(1, -1)
        fs = scaler.transform(flat)
        pred = int(model.predict(fs)[0])
        proba = model.predict_proba(fs)[0]
        return pred, proba
    else:
        # CRITICAL: apply same preprocessing as training
        processed = preprocess_eegnet(sample)
        with torch.no_grad():
            x = torch.tensor(processed, dtype=torch.float32).unsqueeze(0)
            logits = model(x)
            proba = torch.softmax(logits, dim=1).numpy()[0]
            return int(np.argmax(proba)), proba

# ═══════════════════════════════════════════════════════════
# ACTIONS
# ═══════════════════════════════════════════════════════════
if single_btn:
    idx = random.randint(0, len(X_data)-1)
    sample = X_data[idx]
    pred, proba = predict(sample)
    st.session_state.current_sample = sample
    st.session_state.current_true = int(y_data[idx])
    st.session_state.prediction = pred
    st.session_state.confidence = proba
    st.session_state.samples_processed += 1
    entry = {'time': datetime.now().strftime('%H:%M:%S'),
             'pred': label_map[pred], 'conf': float(max(proba))}
    if "Test" in operating_mode:
        entry['true'] = label_map[int(y_data[idx])]
        entry['correct'] = pred == int(y_data[idx])
    else:
        entry['true'] = None; entry['correct'] = None
    st.session_state.session_log.append(entry)

if start_btn:
    st.session_state.playing = True
    if st.session_state.buffer is None:
        st.session_state.buffer = X_data[random.randint(0, len(X_data)-1)].copy()

if st.session_state.playing:
    idx = random.randint(0, len(X_data)-1)
    new_sample = X_data[idx]
    if st.session_state.buffer is None:
        st.session_state.buffer = new_sample.copy()
    else:
        buf = st.session_state.buffer
        shift = 30
        buf = np.roll(buf, -shift, axis=-1)
        new_chunk = new_sample[:, -shift:] + np.random.randn(buf.shape[0], shift)*5
        buf[:, -shift:] = new_chunk
        st.session_state.buffer = buf

    if st.session_state.samples_processed % 3 == 0:
        pred, proba = predict(new_sample)
        st.session_state.prediction = pred
        st.session_state.confidence = proba
        st.session_state.current_true = int(y_data[idx])
        st.session_state.current_sample = new_sample
        entry = {'time': datetime.now().strftime('%H:%M:%S'),
                 'pred': label_map[pred], 'conf': float(max(proba))}
        if "Test" in operating_mode:
            entry['true'] = label_map[int(y_data[idx])]
            entry['correct'] = pred == int(y_data[idx])
        else:
            entry['true'] = None; entry['correct'] = None
        st.session_state.session_log.append(entry)
    st.session_state.samples_processed += 1

if len(st.session_state.session_log) > 20:
    st.session_state.session_log = st.session_state.session_log[-20:]

# ═══════════════════════════════════════════════════════════
# STATUS
# ═══════════════════════════════════════════════════════════
s = '<div class="status-bar">'
if st.session_state.playing: s += '<span class="status-live">🟢 STREAMING</span>'
elif st.session_state.calibrating: s += '<span class="status-cal">🟡 CALIBRATING</span>'
else: s += '<span class="status-idle">🔴 IDLE</span>'
s += f' &nbsp;|&nbsp; Mode: <b>{operating_mode}</b>'
s += f' &nbsp;|&nbsp; Decoder: <b>{mode}</b>'
s += f' &nbsp;|&nbsp; Samples: <b>{st.session_state.samples_processed}</b>'
s += f' &nbsp;|&nbsp; Time: <b>{datetime.now().strftime("%H:%M:%S")}</b>'
s += '</div>'
st.markdown(s, unsafe_allow_html=True)

# ═══════════════════════════════════════════════════════════
# CALIBRATION
# ═══════════════════════════════════════════════════════════
if st.session_state.calibrating:
    st.markdown("---")
    st.markdown("## 🎓 Calibration Session")
    st.info(f"**Collected:** {len(st.session_state.calibration_data)} samples. "
            f"Record each class by holding the thought.")
    classes = ['Rest','Left Fist','Right Fist'] if "Focus" in mode else \
              ['Rest','Left Fist','Right Fist','Both Fists','Both Feet']
    for cls_name in classes:
        c1, c2 = st.columns([3,1])
        with c1: st.markdown(f"**{cls_name}**")
        with c2:
            if st.button(f"Record", key=f"cal_{cls_name}"):
                idx = random.randint(0, len(X_data)-1)
                st.session_state.calibration_data.append((X_data[idx].copy(), cls_name))
                st.rerun()
    c1, c2 = st.columns(2)
    with c1:
        if st.button("✅ Finish", type="primary"):
            st.session_state.calibrating = False
            st.success(f"Calibration complete: {len(st.session_state.calibration_data)} samples.")
    with c2:
        if st.button("❌ Cancel"):
            st.session_state.calibrating = False
            st.session_state.calibration_data = []
            st.rerun()
    st.markdown("---")

# ═══════════════════════════════════════════════════════════
# MAIN UI
# ═══════════════════════════════════════════════════════════
st.markdown(f'<div class="mode-badge {badge_class}">{badge_text}</div>', unsafe_allow_html=True)
col_left, col_right = st.columns([1,1])

with col_left:
    st.markdown("### 📊 Prediction")
    if st.session_state.prediction is not None:
        pred = st.session_state.prediction
        proba = st.session_state.confidence
        pred_name = label_map[pred]
        card_class = "pred-card" if "Focus" in mode else "pred-card pred-card-5"
        pred_color = "#00d4ff" if "Focus" in mode else "#ff9f1c"
        h = f'<div class="{card_class}">'
        h += f'<p class="pred-label">Predicted Intent</p>'
        h += f'<p class="pred-value" style="color:{pred_color};">{pred_name}</p>'
        if "Test" in operating_mode and st.session_state.current_true is not None:
            true_name = label_map[st.session_state.current_true]
            h += f'<p class="true-label">Ground Truth</p>'
            h += f'<p class="true-value">{true_name}</p>'
            h += '<p class="correct">✅ Correct</p>' if pred == st.session_state.current_true else '<p class="incorrect">❌ Incorrect</p>'
        else:
            h += f'<p class="true-label">Mode</p>'
            h += f'<p class="true-value">Live Stream · No ground truth</p>'
        h += '</div>'
        st.markdown(h, unsafe_allow_html=True)
    else:
        st.info("Click **▶ Stream** or **🎲 Sample** to begin.")

with col_right:
    st.markdown("### 📈 Confidence Distribution")
    if st.session_state.confidence is not None:
        proba = st.session_state.confidence
        bar_class = "conf-bar-fill" if "Focus" in mode else "conf-bar-fill conf-bar-fill-5"
        for i, cls in label_map.items():
            if i < len(proba):
                pct = float(proba[i])*100
                st.markdown(f'<div class="conf-row"><div class="conf-name">{cls}</div>'
                            f'<div class="conf-bar-bg"><div class="{bar_class}" style="width:{pct}%;"></div></div>'
                            f'<div class="conf-pct">{pct:.2f}%</div></div>',
                            unsafe_allow_html=True)
    else:
        st.info("Awaiting signal...")

# ═══════════════════════════════════════════════════════════
# EEG GRAPH
# ═══════════════════════════════════════════════════════════
st.markdown("### 🧠 Live EEG Waveforms")
buf = st.session_state.buffer if st.session_state.buffer is not None else st.session_state.current_sample
if buf is not None:
    n_ch = min(6, buf.shape[0])
    fig, axes = plt.subplots(n_ch, 1, figsize=(13,6), sharex=True)
    fig.patch.set_facecolor('#0a0d12')
    if n_ch == 1: axes = [axes]
    for i, ax in enumerate(axes):
        ax.set_facecolor('#0a0d12')
        ax.plot(buf[i], linewidth=0.7, color=accent)
        ax.set_ylabel(f'Ch{i+1}', color='#9ca3af', fontsize=8, rotation=0, labelpad=20)
        ax.tick_params(colors='#9ca3af', labelsize=8)
        ax.grid(True, alpha=0.12, color='#2a2f3a')
        for spine in ax.spines.values(): spine.set_color('#2a2f3a')
    axes[-1].set_xlabel('Time (samples)', color='#9ca3af', fontsize=9)
    plt.tight_layout()
    st.pyplot(fig); plt.close(fig)

    st.markdown("### 📉 Signal Metrics")
    c1,c2,c3,c4 = st.columns(4)
    with c1: st.markdown(f'<div class="stat-box"><div class="stat-value">{buf.min()*1000:.2f}</div><div class="stat-label">Min (mV)</div></div>', unsafe_allow_html=True)
    with c2: st.markdown(f'<div class="stat-box"><div class="stat-value">{buf.max()*1000:.2f}</div><div class="stat-label">Max (mV)</div></div>', unsafe_allow_html=True)
    with c3: st.markdown(f'<div class="stat-box"><div class="stat-value">{buf.std()*1000:.2f}</div><div class="stat-label">Std (mV)</div></div>', unsafe_allow_html=True)
    with c4: st.markdown(f'<div class="stat-box"><div class="stat-value">{buf.shape[0]}</div><div class="stat-label">Channels</div></div>', unsafe_allow_html=True)

# ═══════════════════════════════════════════════════════════
# LOG
# ═══════════════════════════════════════════════════════════
st.markdown("### 📋 Session Log")
if st.session_state.session_log:
    for e in reversed(st.session_state.session_log[-10:]):
        if e['true'] is not None:
            cls = "log-entry-match" if e['correct'] else "log-entry-miss"
            ic = "✅" if e['correct'] else "❌"
            line = f"[{e['time']}] {ic} Pred: {e['pred']} | True: {e['true']} | Conf: {e['conf']*100:.1f}%"
        else:
            cls = "log-entry"
            line = f"[{e['time']}] · Pred: {e['pred']} | Conf: {e['conf']*100:.1f}%"
        st.markdown(f'<div class="log-entry {cls}">{line}</div>', unsafe_allow_html=True)
else:
    st.info("No predictions yet.")

st.markdown("---")
st.markdown('<div class="footer">A818_NF · Brain Signal Decoder · v2.1 · Built by Asma Rizwan Rao</div>', unsafe_allow_html=True)

if st.session_state.playing:
    time.sleep(0.4)
    st.rerun()

2026-10-02 09:15:54.824 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-02 09:15:54.828 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-02 09:15:54.955 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-02 09:15:54.958 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-02 09:15:54.960 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name '__file__' is not defined

In [29]:
!pkill -f streamlit
!rm -rf /content/A818_NF
!git clone https://github.com/deepbuildASI/NeuroFlow.git /content/A818_NF

Cloning into '/content/A818_NF'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 64 (delta 23), reused 45 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 3.45 MiB | 13.42 MiB/s, done.
Resolving deltas: 100% (23/23), done.


In [30]:
!cd /content/A818_NF && streamlit run app.py &>/content/logs.txt & sleep 5 && /content/cloudflared tunnel --url http://localhost:8501

2026-10-02T10:23:58Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-02T10:23:58Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-02T10:24:05Z INF +--------------------------------------------------------------------------------------------+
2026-10-02T10:24:05Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-02T10:24:05Z INF |  https://oct-flexibility-price-deaf.trycloudflare.com 

In [31]:
import numpy as np

DRIVE = '/content/drive/MyDrive/NeuroFlow'

# Slim 3-class data (500 samples per class = 1500 total)
X3 = np.load(f'{DRIVE}/X_all.npy')
y3 = np.load(f'{DRIVE}/y_all.npy')
print("3-class full:", X3.shape)

X3_slim, y3_slim = [], []
for cls in np.unique(y3):
    idx = np.where(y3 == cls)[0]
    n = min(500, len(idx))
    chosen = np.random.choice(idx, n, replace=False)
    X3_slim.append(X3[chosen])
    y3_slim.append(y3[chosen])

X3_slim = np.concatenate(X3_slim)
y3_slim = np.concatenate(y3_slim)
print("3-class slim:", X3_slim.shape)

np.save(f'{DRIVE}/app_data_X3.npy', X3_slim)
np.save(f'{DRIVE}/app_data_y3.npy', y3_slim)

# Slim 5-class data (200 per class = 1000 total)
X5 = np.load(f'{DRIVE}/A818_NF_5class_X_bal.npy')
y5 = np.load(f'{DRIVE}/A818_NF_5class_y_bal.npy')
print("5-class full:", X5.shape)

X5_slim, y5_slim = [], []
for cls in np.unique(y5):
    idx = np.where(y5 == cls)[0]
    n = min(200, len(idx))
    chosen = np.random.choice(idx, n, replace=False)
    X5_slim.append(X5[chosen])
    y5_slim.append(y5[chosen])

X5_slim = np.concatenate(X5_slim)
y5_slim = np.concatenate(y5_slim)
print("5-class slim:", X5_slim.shape)

np.save(f'{DRIVE}/app_data_X5.npy', X5_slim)
np.save(f'{DRIVE}/app_data_y5.npy', y5_slim)

print("\nSaved slim files to Drive")

3-class full: (2820, 64, 481)
3-class slim: (1500, 64, 481)
5-class full: (4500, 64, 481)
5-class slim: (1000, 64, 481)

Saved slim files to Drive


In [32]:
import numpy as np
from google.colab import files

DRIVE = '/content/drive/MyDrive/NeuroFlow'

# Load and compress
for name in ['app_data_X3', 'app_data_X5']:
    X = np.load(f'{DRIVE}/{name}.npy')
    X_half = X.astype(np.float16)
    np.save(f'{DRIVE}/{name}_fp16.npy', X_half)
    import os
    size_mb = os.path.getsize(f'{DRIVE}/{name}_fp16.npy') / 1024 / 1024
    print(f"{name}_fp16.npy — {size_mb:.1f} MB")

print("\nDone. Now downloading...")

app_data_X3_fp16.npy — 88.1 MB
app_data_X5_fp16.npy — 58.7 MB

Done. Now downloading...


In [33]:
from google.colab import files

DRIVE = '/content/drive/MyDrive/NeuroFlow'

downloads = [
    f'{DRIVE}/app_data_X3_fp16.npy',
    f'{DRIVE}/app_data_y3.npy',
    f'{DRIVE}/app_data_X5_fp16.npy',
    f'{DRIVE}/app_data_y5.npy',
    f'{DRIVE}/best_model.pkl',
    f'{DRIVE}/scaler.pkl',
    f'{DRIVE}/A818_NF_5class_eegnet_v2.pth',
]

for path in downloads:
    print(f"Downloading: {path.split('/')[-1]}")
    files.download(path)

print("\nAll downloads started. Check your Downloads folder.")

Downloading: app_data_X3_fp16.npy


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: app_data_y3.npy


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloading: A818_NF_5class_eegnet_v2.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All downloads started. Check your Downloads folder.
